# LIBERO Sim Validation — Minimal 1-Task Closed-Loop Eval

**Scope:** One `libero_spatial` task, zero-shot Lift predictor (auto-fallback to quick LIBERO predictor if transfer is weak), receding-horizon CEM, 20 init states.

**Goal:** Compare embedding-space Success@K (cos ≥ 0.90) against **real MuJoCo task success rate**, and report their correlation.

**Predictor:** `module_a_transformer_K1_T8.pt` (Lift-trained, K=1, T=8) — the same checkpoint used by `module_d_transfer_comparison.ipynb` for Lift→Can/Square transfer.

**Strategy:** Auto-fallback — run zero-shot first; if `val_cos < 0.5`, train a quick LIBERO predictor (~20 epochs) and use that for the closed-loop sim eval instead.

**Outputs:** `outputs/libero_sim_eval.pt`, `outputs/libero_sim_correlation.png`, `outputs/libero_results.tex`

In [6]:
import os, sys

# --- Install LIBERO + deps (Colab Linux) ---
!pip install -q einops timm tqdm scikit-learn matplotlib scipy h5py pillow

!git clone -q https://github.com/Lifelong-Robot-Learning/LIBERO.git /content/LIBERO 2>/dev/null || true
!cd /content/LIBERO && pip install -q -e . 2>&1 | tail -5
!pip install -q "protobuf<4"  # compat fix

# --- MuJoCo rendering backend (must be set before importing mujoco/robosuite) ---
os.environ['MUJOCO_GL'] = 'egl'
os.environ['MUJOCO_EGL_DEVICE_ID'] = '0'

sys.path.insert(0, '/content/LIBERO')
print('Install done. If libero import fails, restart kernel and re-run from Cell 3.')

ON_COLAB=True | LIBERO_ROOT='/content/LIBERO'
Install done. If libero import fails, restart kernel and re-run from Cell 3.


In [7]:
import torch, torch.nn as nn, torch.nn.functional as F
import numpy as np, os, glob, h5py, copy, warnings, shutil
from pathlib import Path
from torch.utils.data import Dataset, DataLoader
from torch.cuda.amp import autocast, GradScaler
from tqdm.auto import tqdm
import matplotlib.pyplot as plt
from PIL import Image
from torchvision import transforms

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')

# --- Config (mirrors module_d_transfer_comparison.ipynb) ---
class Config:
    embed_dim    = 384
    action_dim   = 7
    d_model      = 256
    n_heads      = 4
    n_layers_tr  = 4
    n_layers_sg  = 2
    batch_size   = 64
    lr           = 1e-4
    weight_decay = 0.05
    warmup_steps = 200
    use_amp      = True
    dropout_p    = 0.2
    seed         = 42
    K_values     = [1, 5, 10, 20]
    best_T       = 8
    n_train_episodes = 160
    cem_n_pop        = 128
    cem_elite_frac   = 0.1
    cem_n_iters      = 5
    success_threshold = 0.90
    bc_rollout_H     = 20
    # --- LIBERO-specific ---
    libero_suite     = 'libero_spatial'
    libero_task_id   = 0
    n_eval_episodes  = 20
    cem_h            = 5
    replan_every     = 10
    max_sim_steps    = 200
    K                = 1
    image_size       = 224
    subsample_rate   = 2
    zero_shot_threshold = 0.5   # val_cos below this triggers Plan B
    plan_b_epochs       = 20

cfg = Config()
torch.manual_seed(cfg.seed); np.random.seed(cfg.seed)

OUTPUT_DIR = Path('/content/outputs')
EMBED_DIR  = Path('/content/data/embeddings')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
EMBED_DIR.mkdir(parents=True, exist_ok=True)

Device: cuda


In [8]:
# --- DINOv2 ViT-S/14 (frozen, fp16) — from Can_Square_Hier_experiment.ipynb ---
print('Loading DINOv2 ViT-S/14...')
dinov2 = torch.hub.load('facebookresearch/dinov2', 'dinov2_vits14', trust_repo=True)
dinov2 = dinov2.to(device).eval()
if cfg.use_amp and device.type == 'cuda':
    dinov2 = dinov2.half()
for p in dinov2.parameters():
    p.requires_grad = False
print(f'DINOv2 loaded: {sum(p.numel() for p in dinov2.parameters())/1e6:.1f}M params')

# --- Image transform (128->224, ImageNet norm) ---
dino_transform = transforms.Compose([
    transforms.Resize((cfg.image_size, cfg.image_size)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

def embed_frame(frame_rgb):
    """Single (H,W,3) uint8 -> (384,) tensor on device."""
    img = Image.fromarray(frame_rgb.astype(np.uint8))
    t = dino_transform(img).unsqueeze(0).to(device)
    if cfg.use_amp and device.type == 'cuda':
        with autocast(): emb = dinov2(t)
    else:
        emb = dinov2(t)
    return emb.squeeze(0).float()

Loading DINOv2 ViT-S/14...


Using cache found in /root/.cache/torch/hub/facebookresearch_dinov2_main


DINOv2 loaded: 22.1M params


In [9]:
# --- ActionMLP (from module_a_predictor.ipynb) ---
class ActionMLP(nn.Module):
    def __init__(self, action_dim=7, hidden_dim=128, out_dim=256):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(action_dim, hidden_dim), nn.GELU(),
            nn.Linear(hidden_dim, out_dim),
        )
    def forward(self, a): return self.net(a)

# --- TransformerPredictor (from module_a_predictor.ipynb) ---
class TransformerPredictor(nn.Module):
    def __init__(self, embed_dim=384, action_dim=7, d_model=256,
                 n_layers=4, n_heads=4, max_seq_len=64, dropout=0.2):
        super().__init__()
        self.d_model = d_model
        self.embed_proj = nn.Linear(embed_dim, d_model)
        self.action_mlp = ActionMLP(action_dim, hidden_dim=128, out_dim=d_model)
        self.pos_embed = nn.Parameter(torch.randn(1, max_seq_len, d_model) * 0.02)
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=d_model, nhead=n_heads, batch_first=True,
            dropout=dropout, activation='gelu')
        self.transformer = nn.TransformerEncoder(encoder_layer, n_layers)
        self.predictor = nn.Linear(d_model, embed_dim)
        self._init_weights()

    def _init_weights(self):
        for p in self.parameters():
            if p.dim() > 1: nn.init.xavier_uniform_(p)

    def forward(self, embeds, actions):
        if embeds.dim() == 2:
            embeds = embeds.unsqueeze(1); actions = actions.unsqueeze(1)
        B, T, _ = embeds.shape
        x = self.embed_proj(embeds) + self.action_mlp(actions)
        x = x + self.pos_embed[:, :T, :]
        causal_mask = torch.triu(torch.ones(T, T, device=x.device, dtype=torch.bool), diagonal=1)
        x = self.transformer(x, mask=causal_mask)
        return self.predictor(x[:, -1, :])

# --- Batch embedding (from Can_Square_Hier_experiment.ipynb) ---
@torch.no_grad()
def compute_embeddings(frames_array, batch_size=64, desc='Embedding'):
    n = len(frames_array)
    embeddings = np.zeros((n, cfg.embed_dim), dtype=np.float32)
    for i in tqdm(range(0, n, batch_size), desc=desc):
        batch_frames = frames_array[i:i+batch_size]
        tensors = [dino_transform(Image.fromarray(f.astype(np.uint8))) for f in batch_frames]
        batch = torch.stack(tensors).to(device)
        if cfg.use_amp and device.type == 'cuda':
            with autocast(): emb = dinov2(batch)
        else:
            emb = dinov2(batch)
        embeddings[i:i+batch_size] = emb.float().cpu().numpy()
    return embeddings

In [10]:
def load_predictor(ckpt_path, K=None, T=None):
    """Load a trained JEPA predictor from checkpoint."""
    ckpt_path = Path(ckpt_path)
    if not ckpt_path.exists():
        print(f'Checkpoint not found: {ckpt_path}')
        return None
    ckpt = torch.load(ckpt_path, map_location=device, weights_only=False)
    model = TransformerPredictor(
        embed_dim=cfg.embed_dim, action_dim=cfg.action_dim, d_model=cfg.d_model,
        n_layers=cfg.n_layers_tr, n_heads=cfg.n_heads, dropout=cfg.dropout_p,
    ).to(device)
    model.load_state_dict(ckpt['model_state_dict'])
    model.eval()
    vc = ckpt.get('val_cos', ckpt.get('best_cos', '?'))
    print(f'Loaded {ckpt_path.name}  K={ckpt.get("K","?")} T={ckpt.get("T","?")} val_cos={vc}')
    return model

# --- Upload Lift predictor checkpoint + norm_stats ---
# Method 1: Direct upload (interactive)
try:
    from google.colab import files
    print('Upload: module_a_transformer_K1_T8.pt  AND  norm_stats.pt')
    print('(from local outputs/ and data/embeddings/)')
    uploaded = files.upload()
    for name, data in uploaded.items():
        if 'K1_T8' in name or 'transformer' in name:
            (OUTPUT_DIR / name).write_bytes(data)
        else:
            (EMBED_DIR / name).write_bytes(data)
except ImportError:
    print('Not on Colab. Place files manually:')
    print(f'  {OUTPUT_DIR}/module_a_transformer_K1_T8.pt')
    print(f'  {EMBED_DIR}/norm_stats.pt')

# Alternative: mount Google Drive
# from google.colab import drive; drive.mount('/content/drive')
# shutil.copy('/content/drive/MyDrive/outputs/module_a_transformer_K1_T8.pt', OUTPUT_DIR)
# shutil.copy('/content/drive/MyDrive/data/embeddings/norm_stats.pt', EMBED_DIR)

# Find norm_stats
ns_path = EMBED_DIR / 'norm_stats.pt'
if not ns_path.exists():
    cands = list(EMBED_DIR.glob('norm_stats*.pt'))
    if cands: ns_path = cands[0]
lift_norm_stats = torch.load(ns_path, map_location=device, weights_only=False)
print(f'Lift norm_stats: mean={lift_norm_stats["mean"].cpu().numpy().round(3)}, '
      f'std={lift_norm_stats["std"].cpu().numpy().round(3)}')

lift_predictor = load_predictor(OUTPUT_DIR / 'module_a_transformer_K1_T8.pt')
assert lift_predictor is not None, 'Failed to load Lift predictor!'

Upload: module_a_transformer_K1_T8.pt  AND  norm_stats.pt
(from local outputs/ and data/embeddings/)


KeyboardInterrupt: 

In [ ]:
from libero.libero import benchmark, get_libero_path
from libero.libero.envs import OffScreenRenderEnv

benchmark_dict = benchmark.get_benchmark_dict()
task_suite = benchmark_dict[cfg.libero_suite]()
task = task_suite.get_task(cfg.libero_task_id)
task_name = task.name
task_lang = task.language
bddl_file = os.path.join(get_libero_path('bddl_files'), task.problem_folder, task.bddl_file)
print(f'Task: {task_name}')
print(f'Instruction: {task_lang}')
print(f'BDDL: {bddl_file}')

env_args = {'bddl_file_name': bddl_file, 'camera_heights': 128, 'camera_widths': 128}
env = OffScreenRenderEnv(**env_args)
env.seed(cfg.seed)
env.reset()
init_states = task_suite.get_task_init_states(cfg.libero_task_id)
print(f'Init states available: {len(init_states)}')
env.set_init_state(init_states[0])

# Smoke test: get obs, check image key + shape
obs = env.get_observation()
img_key = 'agentview_image' if 'agentview_image' in obs else 'agentview_rgb'
print(f'Obs image key: {img_key}')
frame0 = obs[img_key]
print(f'Frame shape: {frame0.shape}, dtype: {frame0.dtype}')

# Test embedding
z0 = embed_frame(frame0 if frame0.ndim == 3 and frame0.shape[-1] == 3 else frame0.transpose(1, 2, 0))
print(f'Test embedding shape: {z0.shape}, norm: {z0.norm().item():.3f}')
env.close()
print('LIBERO env smoke test passed.')

In [ ]:
TASK = 'libero_spatial'

# --- Download LIBERO demo dataset ---
!cd /content/LIBERO && python benchmark_scripts/download_libero_datasets.py --datasets libero_spatial --use-huggingface 2>&1 | tail -5

# --- Find demo HDF5 files ---
demo_files = glob.glob('/content/LIBERO/**/libero_spatial/**/*.hdf5', recursive=True)
if not demo_files:
    demo_files = glob.glob('/content/LIBERO/**/*spatial*/**/*.hdf5', recursive=True)
if not demo_files:
    demo_files = glob.glob('/content/LIBERO/**/*.hdf5', recursive=True)
print(f'Found {len(demo_files)} HDF5 files')
for p in demo_files[:5]: print(f'  {p}')
assert len(demo_files) > 0, 'No demo HDF5 found — check download.'

demo_path = demo_files[0]

# --- Load all demo episodes ---
def get_frame_from_obs(obs_dict):
    for key in ['agentview_image', 'agentview_rgb']:
        if key in obs_dict:
            f = obs_dict[key]
            if f.ndim == 3 and f.shape[0] in (3, 4) and f.shape[-1] not in (3, 4):
                f = np.transpose(f, (1, 2, 0))
            return f.astype(np.uint8)
    raise KeyError(f'No image obs found. Keys: {list(obs_dict.keys())}')

all_frames, all_actions, ep_n_frames = [], [], []
with h5py.File(demo_path, 'r') as f:
    demo_keys = sorted(f['data'].keys(), key=lambda k: int(k.split('_')[1]))
    print(f'Episodes in demo file: {len(demo_keys)}')
    for key in tqdm(demo_keys, desc='Loading demos'):
        grp = f['data'][key]
        # Find image key in HDF5
        obs_grp = grp['obs']
        img_key_h5 = 'agentview_image' if 'agentview_image' in obs_grp else 'agentview_rgb'
        imgs = obs_grp[img_key_h5][:]       # (N, H, W, 3) or (N, 3, H, W)
        acts = grp['actions'][:]            # (N, 7)
        # Ensure (N, H, W, 3)
        if imgs.ndim == 4 and imgs.shape[1] in (3, 4):
            imgs = np.transpose(imgs, (0, 2, 3, 1))
        # Subsample 20->10 Hz
        idx = np.arange(0, len(imgs), cfg.subsample_rate)
        all_frames.append(imgs[idx])
        all_actions.append(acts[idx])
        ep_n_frames.append(len(idx))

print(f'Loaded {len(ep_n_frames)} episodes, total frames: {sum(ep_n_frames)}')
print(f'Frame shape: {all_frames[0].shape}, action shape: {all_actions[0].shape}')

In [ ]:
# --- Train/val split (80/20 by episode) ---
n_eps = len(ep_n_frames)
n_train = int(0.8 * n_eps)
train_eps = list(range(n_train))
val_eps   = list(range(n_train, n_eps))
print(f'Train episodes: {len(train_eps)}, Val episodes: {len(val_eps)}')

# Split by episode
train_frames_cat = np.concatenate([all_frames[i] for i in train_eps], axis=0)
val_frames_cat   = np.concatenate([all_frames[i] for i in val_eps], axis=0)
train_acts_cat   = np.concatenate([all_actions[i] for i in train_eps], axis=0)
val_acts_cat     = np.concatenate([all_actions[i] for i in val_eps], axis=0)
print(f'Train frames: {len(train_frames_cat)}, Val frames: {len(val_frames_cat)}')

# --- Episode offsets within each split ---
train_offsets = []; cum = 0
for i in train_eps:
    n = ep_n_frames[i]; train_offsets.append((cum, n)); cum += n
val_offsets = []; cum = 0
for i in val_eps:
    n = ep_n_frames[i]; val_offsets.append((cum, n)); cum += n

# --- Embed ---
emb_train_path = EMBED_DIR / f'embeddings_{TASK}_train.pt'
emb_val_path   = EMBED_DIR / f'embeddings_{TASK}_val.pt'
act_train_path = EMBED_DIR / f'actions_{TASK}_train.pt'
act_val_path   = EMBED_DIR / f'actions_{TASK}_val.pt'

if emb_train_path.exists() and emb_val_path.exists():
    train_embs = torch.load(emb_train_path).float()
    val_embs   = torch.load(emb_val_path).float()
    print('Embeddings already cached.')
else:
    train_embs = compute_embeddings(train_frames_cat, desc='Train emb')
    val_embs   = compute_embeddings(val_frames_cat, desc='Val emb')
    torch.save(torch.tensor(train_embs), emb_train_path)
    torch.save(torch.tensor(val_embs), emb_val_path)

# --- Action normalization (from LIBERO train split) ---
act_mean = train_acts_cat.mean(axis=0).astype(np.float32)
act_std  = train_acts_cat.std(axis=0).astype(np.float32)
act_std  = np.where(act_std < 1e-8, 1.0, act_std)
norm_stats_libero = {'mean': torch.tensor(act_mean), 'std': torch.tensor(act_std)}
torch.save(norm_stats_libero, EMBED_DIR / f'norm_stats_{TASK}.pt')

train_acts_norm = ((train_acts_cat - act_mean) / act_std).astype(np.float32)
val_acts_norm   = ((val_acts_cat   - act_mean) / act_std).astype(np.float32)
torch.save(torch.tensor(train_acts_norm), act_train_path)
torch.save(torch.tensor(val_acts_norm), act_val_path)
print(f'Action stats: mean={act_mean.round(3)}, std={act_std.round(3)}')

# --- Build triplets ---
def build_triplets(offsets, K):
    triplets = []
    for start, length in offsets:
        for t in range(length - K):
            triplets.append([start + t, start + t, start + t + K])
    return torch.tensor(triplets, dtype=torch.long)

K = cfg.K
torch.save(build_triplets(train_offsets, K), EMBED_DIR / f'triplets_{TASK}_train_K{K}.pt')
torch.save(build_triplets(val_offsets, K),   EMBED_DIR / f'triplets_{TASK}_val_K{K}.pt')
print(f'Triplets built for K={K}.')

# --- Eval samples: (z_start, z_goal) per val episode ---
eval_samples = []
for j, (start, n) in enumerate(val_offsets):
    eval_samples.append({
        'ep': val_eps[j],
        'z_start': val_embs[start],
        'z_goal':  val_embs[start + n - 1],
        'n_frames': n,
    })

# Goal embedding for sim: mean of all val demo last-frame embeddings
goal_embs = torch.stack([val_embs[s + n - 1] for s, n in val_offsets])
z_goal_sim = goal_embs.mean(0).to(device)
print(f'Eval samples: {len(eval_samples)}')
print(f'Sim goal embedding: shape={z_goal_sim.shape}, norm={z_goal_sim.norm().item():.3f}')

In [ ]:
# --- Dataset + eval helpers (from module_d_transfer_comparison.ipynb) ---
class ContextTripletDataset(Dataset):
    def __init__(self, embed_path, action_path, triplet_path, T=1):
        self.embeddings = torch.load(embed_path).float()
        self.actions    = torch.load(action_path).float()
        self.triplets   = torch.load(triplet_path)
        self.T = T
        valid = self.triplets[:, 0] >= (T - 1)
        self.triplets = self.triplets[valid]
    def __len__(self): return len(self.triplets)
    def __getitem__(self, idx):
        t_idx, a_idx, tpK_idx = self.triplets[idx]
        z_seq = self.embeddings[t_idx - self.T + 1 : t_idx + 1]
        a_seq = self.actions[t_idx - self.T + 1 : t_idx + 1]
        target = self.embeddings[tpK_idx]
        if z_seq.shape[0] < self.T:
            pad = self.T - z_seq.shape[0]
            z_seq = torch.cat([z_seq[:1].repeat(pad, 1), z_seq])
            a_seq = torch.cat([a_seq[:1].repeat(pad, 1), a_seq])
        return z_seq, a_seq, target

@torch.no_grad()
def compute_val_cosine(model, loader):
    model.eval()
    cs_list = []
    for z_seq, a_seq, target in loader:
        z_seq, a_seq, target = z_seq.to(device), a_seq.to(device), target.to(device)
        if cfg.use_amp and device.type == 'cuda':
            with autocast(): pred = model(z_seq, a_seq)
        else:
            pred = model(z_seq, a_seq)
        cs_list.append(F.cosine_similarity(pred, target, dim=-1).mean().item())
    return float(np.mean(cs_list))

# --- CEM planner (from module_b_hierarchical_planning.ipynb) ---
@torch.no_grad()
def cem_plan(z_start, z_target, predictor, horizon, K, T,
             n_pop=128, elite_frac=0.1, n_iters=5, noise_scale=0.5,
             action_dim=7):
    n_elite = max(1, int(n_pop * elite_frac))
    mu = torch.zeros(horizon, action_dim, device=device)
    sigma = torch.ones(horizon, action_dim, device=device) * noise_scale
    best_seq, best_score = None, -float('inf')
    for it in range(n_iters):
        eps = torch.randn(n_pop, horizon, action_dim, device=device)
        actions = mu.unsqueeze(0) + sigma.unsqueeze(0) * eps
        scores = torch.zeros(n_pop, device=device)
        for i in range(n_pop):
            z_ctx = z_start.clone().unsqueeze(0).unsqueeze(0).repeat(1, T, 1)
            a_ctx = torch.zeros(1, T, action_dim, device=device)
            a_ctx[:, -1, :] = actions[i, 0]
            for step in range(horizon):
                a_cur = actions[i, step:step+1].unsqueeze(0)
                a_ctx = torch.cat([a_ctx[:, 1:, :], a_cur], dim=1)
                pred = predictor(z_ctx, a_ctx)
                z_ctx = torch.cat([z_ctx[:, 1:, :], pred.unsqueeze(1)], dim=1)
            z_pred = z_ctx[:, -1, :].squeeze(0)
            scores[i] = F.cosine_similarity(z_pred.unsqueeze(0), z_target.unsqueeze(0), dim=-1)
        elite_idx = torch.topk(scores, n_elite).indices
        ea = actions[elite_idx]
        mu = ea.mean(dim=0); sigma = ea.std(dim=0) + 1e-6
        iter_best = scores[elite_idx[0]].item()
        if iter_best > best_score:
            best_score = iter_best; best_seq = ea[0].cpu().numpy()
    return best_seq, best_score

# --- Plan A: Zero-shot embedding-space eval ---
K = cfg.K
T = cfg.best_T
horizon = max(1, cfg.bc_rollout_H // K)

val_ds = ContextTripletDataset(
    EMBED_DIR / f'embeddings_{TASK}_val.pt',
    EMBED_DIR / f'actions_{TASK}_val.pt',
    EMBED_DIR / f'triplets_{TASK}_val_K{K}.pt',
    T=T)
val_loader = DataLoader(val_ds, batch_size=cfg.batch_size, shuffle=False,
                        pin_memory=(device.type == 'cuda'))

zs_val_cos = compute_val_cosine(lift_predictor, val_loader)
print(f'Plan A (zero-shot Lift->LIBERO): val_cos = {zs_val_cos:.4f}')

# CEM embedding-space success
cem_scores = []
for sample in tqdm(eval_samples, desc='Zero-shot CEM'):
    _, score = cem_plan(sample['z_start'].to(device), sample['z_goal'].to(device),
                        lift_predictor, horizon, K, T,
                        cfg.cem_n_pop, cfg.cem_elite_frac, cfg.cem_n_iters)
    cem_scores.append(score)
zs_emb_success = np.mean([s >= cfg.success_threshold for s in cem_scores])
zs_goal_cos = np.mean(cem_scores)
print(f'Plan A: emb_success = {zs_emb_success:.2%}, mean goal_cos = {zs_goal_cos:.4f}')

# --- Decision gate ---
use_plan_b = zs_val_cos < cfg.zero_shot_threshold
if use_plan_b:
    print(f'val_cos {zs_val_cos:.4f} < {cfg.zero_shot_threshold} -> triggering Plan B')
else:
    print(f'val_cos {zs_val_cos:.4f} >= {cfg.zero_shot_threshold} -> Plan A sufficient')

In [ ]:
libero_predictor = None
pb_val_cos = None
pb_emb_success = None
pb_goal_cos = None

if use_plan_b:
    print(f'=== Plan B: Training LIBERO predictor ({TASK}, {cfg.plan_b_epochs} epochs) ===')
    train_ds = ContextTripletDataset(
        EMBED_DIR / f'embeddings_{TASK}_train.pt',
        EMBED_DIR / f'actions_{TASK}_train.pt',
        EMBED_DIR / f'triplets_{TASK}_train_K{K}.pt',
        T=T)
    train_loader = DataLoader(train_ds, batch_size=cfg.batch_size, shuffle=True,
                              pin_memory=(device.type == 'cuda'))
    print(f'Train triplets: {len(train_ds)}, Val triplets: {len(val_ds)}')

    libero_predictor = TransformerPredictor(
        embed_dim=cfg.embed_dim, action_dim=cfg.action_dim, d_model=cfg.d_model,
        n_layers=cfg.n_layers_tr, n_heads=cfg.n_heads, dropout=cfg.dropout_p,
    ).to(device)
    opt = torch.optim.AdamW(libero_predictor.parameters(), lr=cfg.lr, weight_decay=cfg.weight_decay)
    scaler = GradScaler(enabled=(cfg.use_amp and device.type == 'cuda'))

    for epoch in range(1, cfg.plan_b_epochs + 1):
        libero_predictor.train()
        losses = []
        for z_seq, a_seq, target in train_loader:
            z_seq, a_seq, target = z_seq.to(device), a_seq.to(device), target.to(device)
            opt.zero_grad()
            if cfg.use_amp and device.type == 'cuda':
                with autocast():
                    pred = libero_predictor(z_seq, a_seq)
                    loss = (1 - F.cosine_similarity(pred, target, dim=-1)).mean()
                scaler.scale(loss).backward(); scaler.step(opt); scaler.update()
            else:
                pred = libero_predictor(z_seq, a_seq)
                loss = (1 - F.cosine_similarity(pred, target, dim=-1)).mean()
                loss.backward(); opt.step()
            losses.append(loss.item())
        val_cos = compute_val_cosine(libero_predictor, val_loader)
        if epoch % 5 == 0 or epoch == 1:
            print(f'  Epoch {epoch:2d}: train_loss={np.mean(losses):.4f} val_cos={val_cos:.4f}')

    pb_val_cos = val_cos
    ckpt_path = OUTPUT_DIR / f'transformer_{TASK}_K{K}_T{T}.pt'
    torch.save({'model_state_dict': libero_predictor.state_dict(),
                'K': K, 'T': T, 'val_cos': val_cos}, ckpt_path)
    print(f'Saved {ckpt_path.name} (val_cos={val_cos:.4f})')

    # Re-eval embedding-space CEM
    pb_cem_scores = []
    for sample in tqdm(eval_samples, desc='Plan B CEM'):
        _, score = cem_plan(sample['z_start'].to(device), sample['z_goal'].to(device),
                            libero_predictor, horizon, K, T,
                            cfg.cem_n_pop, cfg.cem_elite_frac, cfg.cem_n_iters)
        pb_cem_scores.append(score)
    pb_emb_success = np.mean([s >= cfg.success_threshold for s in pb_cem_scores])
    pb_goal_cos = np.mean(pb_cem_scores)
    print(f'Plan B: val_cos={pb_val_cos:.4f}, emb_success={pb_emb_success:.2%}, goal_cos={pb_goal_cos:.4f}')
else:
    print('Plan B skipped (zero-shot sufficient).')

In [ ]:
def libero_sim_rollout(env, init_state, z_goal, predictor, norm_stats,
                       max_steps=200, replan_every=10, cem_h=5, K=1, T=8):
    """Closed-loop rollout in LIBERO MuJoCo sim with receding-horizon CEM.

    Returns: (sim_success, emb_success, goal_cos, n_steps_taken)
    """
    env.reset()
    env.set_init_state(init_state)
    obs = env.get_observation()
    frame = get_frame_from_obs(obs)
    z_cur = embed_frame(frame)

    z_ctx = z_cur.unsqueeze(0).unsqueeze(0).repeat(1, T, 1)
    a_ctx = torch.zeros(1, T, cfg.action_dim, device=device)
    mean = norm_stats['mean'].to(device)
    std  = norm_stats['std'].to(device)

    sim_success = 0
    action_seq = None
    seq_idx = 0

    for t in range(max_steps):
        # Receding-horizon: replan every replan_every steps
        if t % replan_every == 0:
            seq, _ = cem_plan(z_cur, z_goal, predictor, cem_h, K, T,
                              cfg.cem_n_pop, cfg.cem_elite_frac, cfg.cem_n_iters)
            action_seq = torch.tensor(seq, device=device).float()
            seq_idx = 0

        # Get next action from planned sequence
        a_norm = action_seq[seq_idx]
        seq_idx += 1

        # De-normalize: normalized action -> raw action space
        a_raw = (a_norm * std + mean).cpu().numpy()
        a_raw = np.clip(a_raw, -1.0, 1.0)

        # Step sim
        obs, reward, done, info = env.step(a_raw)

        # Embed new observation
        frame = get_frame_from_obs(obs)
        z_cur = embed_frame(frame)

        # Update rolling context
        a_ctx = torch.cat([a_ctx[:, 1:, :], a_norm.view(1, 1, cfg.action_dim)], dim=1)
        z_ctx = torch.cat([z_ctx[:, 1:, :], z_cur.view(1, 1, cfg.embed_dim)], dim=1)

        # Check task completion (LIBERO: sparse reward +1 on success)
        d = done.get('task', False) if isinstance(done, dict) else done
        if d or reward > 0:
            sim_success = 1
            break

    goal_cos = F.cosine_similarity(
        z_cur.view(1, -1), z_goal.view(1, -1).to(device), dim=-1).item()
    emb_success = 1.0 if goal_cos >= cfg.success_threshold else 0.0
    return sim_success, emb_success, goal_cos, t + 1

print('libero_sim_rollout defined.')

In [ ]:
# --- Select predictor + norm_stats for closed-loop ---
if use_plan_b and libero_predictor is not None:
    sim_predictor   = libero_predictor
    sim_norm_stats  = norm_stats_libero
    plan_label      = 'Plan B (LIBERO-trained)'
else:
    sim_predictor   = lift_predictor
    sim_norm_stats  = lift_norm_stats
    plan_label      = 'Plan A (zero-shot Lift)'

print(f'Closed-loop eval with: {plan_label}')
print(f'  Episodes:    {cfg.n_eval_episodes}')
print(f'  max_steps:   {cfg.max_sim_steps}')
print(f'  replan_every:{cfg.replan_every}')
print(f'  cem_h:       {cfg.cem_h}')
print(f'  K={cfg.K}, T={cfg.best_T}')

env = OffScreenRenderEnv(**env_args)
env.seed(cfg.seed)

results = []
n_eval = min(cfg.n_eval_episodes, len(init_states))
for ep_idx in tqdm(range(n_eval), desc='Sim rollout'):
    init_state = init_states[ep_idx]
    sim_succ, emb_succ, goal_cos, n_steps = libero_sim_rollout(
        env, init_state, z_goal_sim, sim_predictor, sim_norm_stats,
        max_steps=cfg.max_sim_steps, replan_every=cfg.replan_every,
        cem_h=cfg.cem_h, K=cfg.K, T=cfg.best_T)
    results.append({
        'ep': ep_idx, 'sim_success': sim_succ,
        'emb_success': emb_succ, 'goal_cos': goal_cos, 'n_steps': n_steps,
    })
    if (ep_idx + 1) % 5 == 0:
        print(f'  Ep {ep_idx+1:2d}: sim={sim_succ} emb={emb_succ} '
              f'cos={goal_cos:.3f} steps={n_steps}')

env.close()
print(f'\nDone. Sim success: {sum(r["sim_success"] for r in results)}/{len(results)}')

In [ ]:
from scipy.stats import spearmanr, pearsonr

# --- Aggregate ---
sim_successes = [r['sim_success'] for r in results]
emb_successes = [r['emb_success'] for r in results]
goal_coss     = [r['goal_cos'] for r in results]
n_steps_list  = [r['n_steps'] for r in results]

sim_rate  = np.mean(sim_successes)
emb_rate  = np.mean(emb_successes)
mean_cos  = np.mean(goal_coss)

# Correlation
if len(set(sim_successes)) > 1:
    rho_s, p_s = spearmanr(goal_coss, sim_successes)
    rho_p, p_p = pearsonr(goal_coss, sim_successes)
else:
    rho_s = rho_p = float('nan')
    p_s = p_p = 1.0

# Embedding-space results from earlier
if use_plan_b and pb_val_cos is not None:
    emb_plan       = 'Plan B'
    val_cos_report = pb_val_cos
    emb_cem_succ   = pb_emb_success
    emb_cem_cos    = pb_goal_cos
else:
    emb_plan       = 'Plan A'
    val_cos_report = zs_val_cos
    emb_cem_succ   = zs_emb_success
    emb_cem_cos    = zs_goal_cos

# --- Results table ---
print('=' * 85)
print(f'LIBERO Sim Validation Results')
print(f'  Suite: {cfg.libero_suite} | Task {cfg.libero_task_id}: {task_name}')
print(f'  Instruction: {task_lang}')
print('=' * 85)
print(f'{"Predictor":<22} {"val_cos":>8} {"Emb Succ@K":>12} {"Sim Succ":>10} '
      f'{"mean cos":>10} {"rho(spear)":>11} {"r(pear)":>9}')
print('-' * 85)
print(f'{plan_label:<22} {val_cos_report:>8.4f} {emb_cem_succ:>12.2%} {sim_rate:>10.2%} '
      f'{mean_cos:>10.4f} {rho_s:>11.3f} {rho_p:>9.3f}')
print('=' * 85)
print(f'Per-episode sim_success: {sim_successes}')
print(f'Per-episode goal_cos:    {[round(c, 3) for c in goal_coss]}')
print(f'Episodes: {len(results)}, Avg steps: {np.mean(n_steps_list):.1f}')

# --- Correlation plot ---
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# Plot 1: per-episode goal_cos, colored by sim success
ax = axes[0]
colors = ['green' if s else 'red' for s in sim_successes]
ax.bar(range(len(goal_coss)), goal_coss, color=colors, alpha=0.7)
ax.axhline(y=cfg.success_threshold, color='blue', linestyle='--',
           label=f'threshold={cfg.success_threshold}')
ax.set_xlabel('Episode'); ax.set_ylabel('Goal cosine sim')
ax.set_title(f'Per-episode goal_cos (green=sim success)\nSim rate: {sim_rate:.2%}')
ax.legend()

# Plot 2: scatter goal_cos vs sim_success + logistic fit
ax = axes[1]
ax.scatter(goal_coss, sim_successes, c=colors, alpha=0.7, s=80, zorder=5)
ax.set_xlabel('Goal cosine sim'); ax.set_ylabel('Sim success (0/1)')
corr_str = f'Spearman rho={rho_s:.3f} (p={p_s:.3f})\nPearson r={rho_p:.3f} (p={p_p:.3f})'
ax.set_title(f'Correlation\n{corr_str}')
if len(set(sim_successes)) > 1:
    try:
        from sklearn.linear_model import LogisticRegression
        X = np.array(goal_coss).reshape(-1, 1)
        y = np.array(sim_successes)
        lr = LogisticRegression().fit(X, y)
        xs = np.linspace(min(goal_coss) - 0.02, max(goal_coss) + 0.02, 100)
        ys = lr.predict_proba(xs.reshape(-1, 1))[:, 1]
        ax.plot(xs, ys, 'b-', alpha=0.5, label='logistic fit')
        ax.legend()
    except Exception:
        pass

plt.tight_layout()
fig_path = OUTPUT_DIR / 'libero_sim_correlation.png'
plt.savefig(fig_path, dpi=150, bbox_inches='tight')
plt.show()
print(f'Figure saved: {fig_path}')

In [ ]:
# --- Save results dict ---
results_dict = {
    'task_suite':      cfg.libero_suite,
    'task_id':         cfg.libero_task_id,
    'task_name':       task_name,
    'task_language':   task_lang,
    'plan':            plan_label,
    'n_eval_episodes': len(results),
    'val_cos':         float(val_cos_report),
    'emb_cem_success': float(emb_cem_succ),
    'emb_cem_goal_cos':float(emb_cem_cos),
    'sim_success_rate':float(sim_rate),
    'mean_goal_cos':   float(mean_cos),
    'spearman_rho':    float(rho_s) if not np.isnan(rho_s) else None,
    'pearson_r':       float(rho_p) if not np.isnan(rho_p) else None,
    'per_episode':     results,
}
rpath = OUTPUT_DIR / 'libero_sim_eval.pt'
torch.save(results_dict, rpath)
print(f'Results saved: {rpath}')

# --- LaTeX one-liner for paper appendix ---
latex = (
    f"% LIBERO sim validation (appendix)\n"
    f"% Task: {task_name} | Predictor: {plan_label}\n"
    f"\\newcommand{{\\liberoTask}}{{{task_name}}}\n"
    f"\\newcommand{{\\liberoSimRate}}{{{sim_rate:.2%}}}\n"
    f"\\newcommand{{\\liberoEmbRate}}{{{emb_cem_succ:.2%}}}\n"
    f"\\newcommand{{\\liberoRho}}{{{rho_s:.3f}}}\n"
    f"\\newcommand{{\\liberoValCos}}{{{val_cos_report:.4f}}}\n"
)
tex_path = OUTPUT_DIR / 'libero_results.tex'
with open(tex_path, 'w') as f:
    f.write(latex)
print(f'LaTeX saved: {tex_path}')

# --- Summary ---
print('\n' + '=' * 60)
print('SUMMARY')
print('=' * 60)
print(f'  Task:        {task_name}')
print(f'  Instruction: {task_lang}')
print(f'  Predictor:   {plan_label}')
print(f'  val_cos:     {val_cos_report:.4f}')
print(f'  Emb success: {emb_cem_succ:.2%} (cos >= {cfg.success_threshold})')
print(f'  Sim success: {sim_rate:.2%} (real MuJoCo task completion)')
print(f'  Correlation: Spearman rho = {rho_s:.3f} (p={p_s:.3f})')
print(f'               Pearson r   = {rho_p:.3f} (p={p_p:.3f})')
print('=' * 60)
print('\nNext: copy outputs/libero_sim_eval.pt + libero_sim_correlation.png')
print('to your local repo for inclusion in module_d_transfer_comparison.ipynb')